## Introduction to Unit Tests

A **unit test** is a small, automated check that verifies whether part of a program behaves as expected.

For example:

```python
def add(a, b):
    return a + b


def test_add():
    assert add(2, 3) == 5
```

The `assert` statement defines the expected result. If the condition is true, the test passes. If it is false, the test fails.

Unit tests help to:

* detect mistakes early;
* document expected behaviour;
* make future code changes safer;
* prevent old bugs from returning.

In this workshop, you will use `pytest` to test whether an experiment can be saved to disk and loaded again without losing important data.

Run the tests with:

```bash
pytest
```

**A test turns “I think it works” into “I checked that it works.”**


In [1]:
# Install pytest
!pip install pytest

  Using cached pytest-9.1.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
Using cached pytest-9.1.1-py3-none-any.whl (386 kB)
Using cached iniconfig-2.3.0-py3-none-any.whl (7.5 kB)
Using cached pluggy-1.6.0-py3-none-any.whl (20 kB)
  Attempting uninstall: pluggy
    Found existing installation: pluggy 1.3.0
    Uninstalling pluggy-1.3.0:
      Successfully uninstalled pluggy-1.3.0


In [6]:
%%writefile test_experiment.py
## Write some tests to a file
def test_add():
    assert 2 + 3 == 5
def test_fail():
    assert 0==0

Overwriting test_experiment.py


In [5]:
# Run pytest on a specific file
!pytest -v test_experiment.py

============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.6.0 -- /opt/conda/bin/python3.11
cachedir: .pytest_cache
rootdir: /home/jovyan/GIT/DataScienceWorkshop/HandsOn
plugins: anyio-4.0.0
collected 2 items                                                              

test_experiment.py::test_add PASSED                                      [ 50%]
test_experiment.py::test_fail FAILED                                     [100%]

=================================== FAILURES ===================================
__________________________________ test_fail ___________________________________

    def test_fail():
>       assert 1==0
E       assert 1 == 0

test_experiment.py:5: AssertionError
=========================== short test summary info ============================
FAILED test_experiment.py::test_fail - assert 1 == 0
========================= 1 failed, 1 passed in 0.06s ==========================


## Unit Testing the Save-and-Load Workflow

Saving an experiment without checking that it can be loaded correctly is risky. A file may be created successfully while still containing incomplete, changed, or incorrectly reconstructed data.

Write a unit test that verifies the complete save-and-load round trip:

```text
Original experiment → Save to disk → Load from disk → Compare
```

The test should:

1. create an example experiment with known values;
2. save the experiment to a temporary file;
3. load the experiment from that file;
4. check that the loaded object contains the expected data.

### Core Challenge: Testing JSON Serialization

For the beginner implementation, create an experiment containing simple JSON-compatible values such as strings, numbers, booleans, lists, and optional values.

Test that important fields are unchanged after saving and loading:

```python
def test_save_and_load_experiment(tmp_path):
    original = Experiment(
        experiment_id="cake-001",
        observer="Ada",
        cake_name="Chocolate Cake",
        rating=9.5,
    )

    file_path = tmp_path / "experiment.json"

    original.save_to_file(file_path)
    loaded = Experiment.load_from_file(file_path)

    assert loaded.experiment_id == original.experiment_id
    assert loaded.observer == original.observer
    assert loaded.cake_name == original.cake_name
    assert loaded.rating == original.rating
```

When using `pytest`, the `tmp_path` fixture provides a temporary directory for the test. This avoids writing test files into the project directory and ensures that each test starts with a clean location.

Depending on the design of your dataclasses, you may be able to compare the complete objects directly:

```python
assert loaded == original
```

### Advanced Challenge: Testing NumPy Arrays

NumPy arrays should not normally be compared with the standard `==` operator. Array comparison produces an array of Boolean values rather than one Boolean result.

Use NumPy’s testing utilities instead:

```python
import numpy as np


def test_save_and_load_numpy_experiment(tmp_path):
    original = Experiment(
        experiment_id="cake-002",
        observer="Grace",
        measurements=np.array([12.1, 12.4, 12.3]),
    )

    file_path = tmp_path / "experiment.json"

    original.save_json(file_path)
    loaded = Experiment.load_json(file_path)

    assert loaded.experiment_id == original.experiment_id
    assert loaded.observer == original.observer

    np.testing.assert_array_equal(
        loaded.measurements,
        original.measurements,
    )
```

For floating-point values that may change slightly during processing, consider:

```python
np.testing.assert_allclose(
    loaded.measurements,
    original.measurements,
)
```

Also consider testing whether the following properties survive serialization:

* array shape;
* array data type;
* number of dimensions;
* measurement units;
* axis descriptions.

For example:

```python
assert loaded.measurements.shape == original.measurements.shape
assert loaded.measurements.dtype == original.measurements.dtype
```

### Professional Challenge: Testing Image Data

For experiments containing images, define what it means for the loaded image to be correct.

Depending on your design, the test could verify:

* that the image path is restored correctly;
* that the referenced image file exists;
* that the image dimensions are unchanged;
* that the pixel data is unchanged;
* that the image data type and number of channels are preserved;
* that image metadata is reconstructed correctly.

When images are represented as NumPy arrays, use NumPy’s array-testing functions:

```python
np.testing.assert_array_equal(
    loaded.image.data,
    original.image.data,
)
```

When images are stored as separate files, the test should use temporary files and directories. Avoid depending on images stored elsewhere on a specific computer, because this would make the test difficult to reproduce.

### Test More Than the Happy Path

A good test suite should not only verify that valid experiments work. It should also examine how the framework behaves when something goes wrong.

Possible additional tests include:

* loading a file that does not exist;
* loading invalid JSON;
* loading a file with a missing required field;
* loading a file with an unsupported format version;
* saving to an invalid location;
* loading an array with the wrong shape;
* loading an experiment whose external image file is missing.

Use `pytest.raises()` when an error is expected:

```python
import pytest


def test_load_missing_file(tmp_path):
    missing_file = tmp_path / "does_not_exist.json"

    with pytest.raises(FileNotFoundError):
        Experiment.load_json(missing_file)
```

### Questions to Consider

While writing your tests, discuss the following:

* Which fields must be identical after loading?
* How should floating-point values be compared?
* How should NumPy arrays be compared?
* Should generated timestamps be excluded from direct comparison?
* Should the test check the contents of the JSON file as well as the loaded object?
* What exceptions should the loading function raise for invalid files?
* How can tests avoid leaving files behind?
* What should happen when external array or image files are missing?
* How can older file-format versions be tested?

## Expected Outcome

At minimum, your project should contain one automated test proving that a representative experiment can be saved and loaded without losing important information.

Run the test with:

```bash
pytest
```

A passing test provides evidence that your serialization workflow currently works. It does not prove that the format will work forever, but it gives you a reliable warning when future code changes accidentally break it.

**Data is not successfully saved until loading it has been tested.**


In [7]:
%%writefile test_dataclass.py

def test_save_and_load_experiment(tmp_path):
    original = Experiment(
        experiment_id="cake-001",
        observer="Ada",
        cake_name="Chocolate Cake",
        rating=9.5,
    )

    file_path = tmp_path / "experiment.json"

    original.save_to_file(file_path)
    loaded = Experiment.load_from_file(file_path)

    assert loaded.experiment_id == original.experiment_id
    assert loaded.observer == original.observer
    assert loaded.cake_name == original.cake_name
    assert loaded.rating == original.rating

Overwriting test_dataclass.py


In [8]:
# Run pytest on a specific file
!pytest -v test_dataclass.py

============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.6.0 -- /opt/conda/bin/python3.11
cachedir: .pytest_cache
rootdir: /home/jovyan/GIT/DataScienceWorkshop/HandsOn
plugins: anyio-4.0.0
collected 1 item                                                               

test_dataclass.py::test_save_and_load_experiment FAILED                  [100%]

=================================== FAILURES ===================================
________________________ test_save_and_load_experiment _________________________

tmp_path = PosixPath('/tmp/pytest-of-jovyan/pytest-0/test_save_and_load_experiment0')

    def test_save_and_load_experiment(tmp_path):
>       original = Experiment(
                   ^^^^^^^^^^
            experiment_id="cake-001",
            observer="Ada",
            cake_name="Chocolate Cake",
            rating=9.5,
        )
E       NameError: name 'Experiment' is not defined

test_dataclass.p